In [3]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import random

#常用 User-Agent 列表
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
]

#硬度映射
BAR_VALUE_MAP = {"40.png": 4, "60.png": 3, "80.png": 2}

def count_snowflakes(td_element):
    """量化雪花分"""
    score = 0.0
    for img in td_element.find_all('img'):
        src = img.get('src', '').lower()
        if "half" in src:
            score += 0.5
        elif "empty" in src:
            continue
        elif "snowflake" in src:
            score += 1.0
    return score

def main():
    #API URL
    api_url = "https://thegoodride.com/ajax.php?mens=0&womens=0"
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "application/json, text/javascript, */*; q=0.01", #模拟XHR请求
        "X-Requested-With": "XMLHttpRequest", #声明这是一个AJAX请求
        "Referer": "https://thegoodride.com/",  #添加来源，更像真实浏览器
        "Accept-Language": "en-US,en;q=0.9"  #添加语言偏好
    }

    print("Fetching snowboard list from API...")
    try:
        #请求API，获取JSON格式的原始数据
        res = requests.get(api_url, headers=headers, timeout=15)
        # 检查请求是否成功
        if res.status_code != 200:
            print(f"API request failed with status code: {res.status_code}")
            return
        #尝试解析JSON
        try:
            all_boards_data = res.json()
        except json.JSONDecodeError:
            print("Response is not valid JSON.")
            print("First 500 characters of response:", res.text[:500])
            return

        print(f"Successfully fetched {len(all_boards_data)} boards from API.")

        results = []
        #遍历每块雪板的数据
        for board in all_boards_data:
            try:
                #解析原始数组数据
                brand = board[0] if len(board) > 0 else "N/A"
                name = board[1] if len(board) > 1 else "N/A"
                price_str = board[3] if len(board) > 3 else "0"
                board_id = board[4] if len(board) > 4 else None

                #清理价格并转换为整数
                usd_price = int("".join(re.findall(r'\d+', str(price_str))) or 0)

                #使用ID构造详情页URL
                clean_name = re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '-')
                clean_brand = re.sub(r'[^\w\s-]', '', brand).strip().replace(' ', '-')
                
                if board_id:
                    #使用ID构造链接
                    detail_url = f"https://thegoodride.com/snowboard-reviews/{clean_brand}-{clean_name}-{board_id}/"
                else:
                    # 如果没有ID，可能需要跳过或尝试其他方式
                    print(f"Skipping '{brand} {name}': missing board ID")
                    continue

                print(f"   分析中: {brand} {name} (ID: {board_id})...")

                #进入详情页获取核心性能数据
                d_res = requests.get(detail_url, headers=headers, timeout=10)
                if d_res.status_code != 200:
                    print(f"Failed to fetch detail page: {detail_url}")
                    continue
                    
                d_soup = BeautifulSoup(d_res.text, 'html.parser')
                
                specs = {}
                for row in d_soup.find_all('tr'):
                    #抓取星级评分
                    label = row.find('span', class_='rating-align-left')
                    if label:
                        td_cells = row.find_all('td')
                        if len(td_cells) > 1:
                            specs[label.get_text(strip=True)] = count_snowflakes(td_cells[1])
                    #抓取 Flex 硬度
                    if row.td and "Flex" in row.td.get_text():
                        img = row.find('div', class_='rating-wrapper').find('img')
                        if img and "src" in img.attrs:
                            m = re.search(r'(\d+)\.png', img['src'])
                            if m: specs['Flex_Val'] = BAR_VALUE_MAP.get(m.group(0), 3)

                #性能量化
                pw = specs.get('Powder', 0)
                cv = specs.get('Carving', 0)
                fs = (specs.get('Buttering', 0) + specs.get('Jumps', 0) + specs.get('Flex_Val', 3)) / 3
                
                #性价比指数
                value_deal = round((pw + cv + fs) / (usd_price / 100), 2) if usd_price > 0 else 0

                results.append({
                    "id": board_id,
                    "brand": brand,
                    "name": name,
                    "price_usd": usd_price,
                    "performance": {
                        "powder": pw,
                        "carving": cv,
                        "freestyle": round(fs, 1)
                    },
                    "value_deal": value_deal
                })
                
                time.sleep(0.5)

            except Exception as e:
                print(f"Error processing board: {e}")
                continue

        #最终排序：按性价比排序
        results.sort(key=lambda x: x['value_deal'], reverse=True)

        #保存结果
        with open('snowboards_from_api.json', 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        
        print(f"\n Analysis complete! Processed {len(results)} snowboards.")
        print(f"Results saved to '{output_file}'")

    except Exception as e:
        print(f"Runtime error: {e}")

if __name__ == "__main__":
    main()

Fetching snowboard list from API...
Successfully fetched 683 boards from API.
   分析中: Telos  BackSlash (ID: 96007)...
   分析中: K2 Excavator (ID: 97025)...
   分析中: Stranda Makrill (ID: 108296)...
   分析中: Capita Spring Break Stairmaster (ID: 106940)...
   分析中: Capita Spring Break Powder Glider (ID: 8122)...
   分析中: K2 Sky Pilot (ID: 108130)...
   分析中: K2 Almanac (ID: 106939)...
   分析中: Burton Smooth Operator  (ID: 107856)...
   分析中: Stone Farther (ID: 106937)...
   分析中: Burton Smooth Operator (ID: 106941)...
   分析中: Amplid Time Machine (ID: 106942)...
   分析中: Amplid Souly Grail (ID: 100585)...
   分析中: YES All In XTRM (ID: 106935)...
   分析中: Weston Gnarnia (ID: 106520)...
   分析中: YES Greats (ID: 444)...
   分析中: Jones Women's Howler (ID: 106844)...
   分析中: YES Standard (ID: 3782)...
   分析中: Nidecker Megalight (ID: 345)...
   分析中: Jones Howler (ID: 106653)...
   分析中: Jones Frontier (ID: 8350)...
   分析中: YES Pick Your Line Xtrm (ID: 106637)...
   分析中: Stranda Descender Evo (ID: 106518)...
   

KeyboardInterrupt: 